[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C41_Deep_RL_Course/02_ppo_sac/02_ppo_sac.ipynb)

# 02 · 策略优化：PPO 与 SAC（纯 numpy，从零）

目标：从零手写 **numpy PPO**——策略网 + 价值网 + GAE + clipped surrogate——在 **numpy CartPole** 上把存活步数从随机的 ~20 提到 **150+**；再实现 **SAC 三大件**（tanh 压缩高斯 + log-prob 修正、双 Q 取 min、温度自调 / 软更新）。

路线：策略网采样 → GAE → PPO clip 目标 → **完整 PPO 训练** → SAC 压缩高斯 → SAC 双 Q + 温度 → ✏️ 练习(clip目标/GAE/熵正则/软更新) → 📖 答案 → 🧪 真实超参胶囊。

> 心智模型：**策略梯度内核 = ∇log π · 优势；PPO = 给它装信赖域刹车(clip)；SAC = off-policy + 最大熵**。

## 1 · 策略网络与采样（离散）

策略网是个小 MLP：状态 → 动作 logits → softmax 概率。采样动作并记录 `log π(a|s)`（PPO 要用旧 log-prob）。
我们用 `tanh` 隐层（PG 常用，梯度温和）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def mlp_init(din, h, dout, seed):
    r = np.random.default_rng(seed)
    return [r.normal(0, np.sqrt(2/din), (h, din)), np.zeros(h),
            r.normal(0, np.sqrt(1/h), (dout, h)), np.zeros(dout)]

def mlp_forward(params, X):
    W1, b1, W2, b2 = params
    z1 = X @ W1.T + b1
    h = np.tanh(z1)
    out = h @ W2.T + b2
    return out, (X, z1, h)

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=-1, keepdims=True)

policy = mlp_init(4, 64, 2, seed=0)          # 4维状态(CartPole), 2动作
s = rng.standard_normal((3, 4))
logits, _ = mlp_forward(policy, s)
probs = softmax(logits)
assert probs.shape == (3, 2)
assert np.allclose(probs.sum(1), 1.0), 'softmax 每行和为 1'
# 采样一个动作 + 其 log-prob
a = int(rng.choice(2, p=probs[0]))
logp = np.log(probs[0, a] + 1e-8)
print('probs[0]=', np.round(probs[0], 3), ' 采样动作=', a, ' logπ=', round(logp, 3))
print('✅ 策略网前向 + 采样 + log-prob 正确')

## 2 · GAE：优势估计（复用 C13）

`δ_t = r_t + γV(s_{t+1})(1-done) - V(s_t)`；`Â_t = Σ_l (γλ)^l δ_{t+l}`（反向递推）。
回报目标 `R̂_t = Â_t + V(s_t)` 用来训 critic。这是 C13 学过的，这里实现它供 PPO 用。

In [ ]:
def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    '''values 长度 = T+1（含 bootstrap 的 V(s_T)）。返回 (adv, returns)，长度 T。'''
    T = len(rewards)
    adv = np.zeros(T); last = 0.0
    for t in reversed(range(T)):
        nonterm = 1.0 - dones[t]
        delta = rewards[t] + gamma * values[t + 1] * nonterm - values[t]
        last = delta + gamma * lam * nonterm * last
        adv[t] = last
    returns = adv + values[:T]
    return adv, returns

# 自检：一条全 +1 奖励、值全 0 的轨迹，优势应为正且单调（越早累积越多）
rew = np.ones(5); val = np.zeros(6); done = np.zeros(5); done[-1] = 1.0
adv, ret = compute_gae(rew, val, done, gamma=0.99, lam=0.95)
print('adv =', np.round(adv, 3))
print('returns =', np.round(ret, 3))
assert np.all(adv > 0), '全正奖励下优势应为正'
assert adv[0] > adv[-1], '越早的步累积的折扣优势越大'
# lam=0 时应退化为单步 TD 误差
adv0, _ = compute_gae(rew, val, done, gamma=0.99, lam=0.0)
assert np.allclose(adv0, rew + 0.99*val[1:]*(1-done) - val[:5]), 'lam=0 应为单步TD'
print('✅ GAE 正确（lam 在偏差/方差间插值，lam=0 退化单步 TD）')

## 3 · PPO 的 clipped surrogate

`L_CLIP = min(r·Â, clip(r, 1-ε, 1+ε)·Â)`，`r = exp(logπ_new - logπ_old)`。

我们实现它并验证关键性质：**优势>0 时把比值推高被封顶；优势<0 时把比值压低被托底**——即裁剪移除了过大更新的激励。

In [ ]:
def ppo_clip_objective(logp_new, logp_old, adv, eps=0.2):
    '''返回每个样本的 clipped surrogate（要最大化它，即最小化其负）。'''
    ratio = np.exp(logp_new - logp_old)
    unclipped = ratio * adv
    clipped = np.clip(ratio, 1 - eps, 1 + eps) * adv
    return np.minimum(unclipped, clipped), ratio

logp_old = np.array([-0.7, -0.7, -0.7, -0.7])
adv = np.array([1.0, 1.0, -1.0, -1.0])         # 两好两坏
# 情形：策略把比值推到 1.5（远超 1+eps=1.2）
logp_new = logp_old + np.log(1.5)
obj, ratio = ppo_clip_objective(logp_new, logp_old, adv, eps=0.2)
print('ratio =', np.round(ratio, 2))
print('clipped surrogate =', np.round(obj, 3))
# 好动作(adv>0)被封顶在 1.2*1=1.2；坏动作(adv<0)的 min 取更小(更负)的未裁剪 1.5*(-1)=-1.5
assert np.isclose(obj[0], 1.2) and np.isclose(obj[1], 1.2), '好动作目标被裁剪封顶 1.2'
assert np.isclose(obj[2], -1.5) and np.isclose(obj[3], -1.5), '坏动作取未裁剪(更悲观)'
print('✅ clip 对好动作封顶(不奖励过大更新)、对坏动作取悲观下界 —— 信赖域的一阶近似')

## 4 · 完整 PPO：在 numpy CartPole 上训练

**本模块高潮**：组装 numpy CartPole + 策略/价值网 + GAE + clip + 优势归一化 + 熵正则，训练。

存活步数应从随机的 ~20 爬到 **150+**（固定 seed 可复现，CartPole 满分 200）。

In [ ]:
class CartPole:
    '''numpy 版 CartPole：4维状态[x,xd,θ,θd]，2动作(左/右推)，每步+1，倒下或200步终止。'''
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
        self.g=9.8; self.mp=0.1; self.mt=1.1; self.l=0.5; self.fmag=10.0; self.tau=0.02
        self.x_thr=2.4; self.th_thr=12*2*np.pi/360
    def reset(self):
        self.state = self.rng.uniform(-0.05, 0.05, 4); self.steps = 0
        return self.state.copy()
    def step(self, a):
        x, xd, th, thd = self.state
        force = self.fmag if a == 1 else -self.fmag
        ct, st = np.cos(th), np.sin(th)
        temp = (force + self.mp*self.l*thd**2*st) / self.mt
        thacc = (self.g*st - ct*temp) / (self.l*(4/3 - self.mp*ct**2/self.mt))
        xacc = temp - self.mp*self.l*thacc*ct/self.mt
        x+=self.tau*xd; xd+=self.tau*xacc; th+=self.tau*thd; thd+=self.tau*thacc
        self.state = np.array([x, xd, th, thd]); self.steps += 1
        done = abs(x)>self.x_thr or abs(th)>self.th_thr or self.steps>=200
        return self.state.copy(), 1.0, done

def adam_make(p): return [[np.zeros_like(w) for w in p], [np.zeros_like(w) for w in p], [0]]
def adam_step(p, grads, st, lr):
    m, v, t = st; t[0] += 1
    for i, g in enumerate(grads):
        m[i] = 0.9*m[i] + 0.1*g; v[i] = 0.999*v[i] + 0.001*g*g
        mh = m[i]/(1-0.9**t[0]); vh = v[i]/(1-0.999**t[0])
        p[i] -= lr * mh/(np.sqrt(vh)+1e-8)

In [ ]:
def train_ppo(seed=0, iters=30, steps_per_iter=1500, epochs=10, eps=0.2, beta_ent=0.01):
    np.random.seed(seed)
    env = CartPole(seed=seed)
    pol = mlp_init(4, 64, 2, seed); val = mlp_init(4, 64, 1, seed+1)
    pa, va = adam_make(pol), adam_make(val)
    erng = np.random.default_rng(seed+99)
    hist = []
    for it in range(iters):
        S, A, R, D, LOGP, V = [], [], [], [], [], []
        ep_rets = []; n = 0
        while n < steps_per_iter:
            s = env.reset(); ep_r = 0
            while True:
                logits, _ = mlp_forward(pol, s[None,:]); pr = softmax(logits)[0]
                a = int(erng.choice(2, p=pr)); lp = np.log(pr[a]+1e-8)
                vv, _ = mlp_forward(val, s[None,:])
                s2, r, done = env.step(a)
                S.append(s); A.append(a); R.append(r); D.append(float(done)); LOGP.append(lp); V.append(vv[0,0])
                s = s2; ep_r += r; n += 1
                if done: break
            ep_rets.append(ep_r)
        S=np.array(S); A=np.array(A); R=np.array(R); D=np.array(D); LOGP=np.array(LOGP); V=np.array(V)
        adv, ret = compute_gae(R, np.append(V, 0.0), D)
        adv = (adv - adv.mean())/(adv.std()+1e-8)            # 优势归一化(关键细节!)
        for _ in range(epochs):
            logits, cache = mlp_forward(pol, S); pr = softmax(logits)
            lp = np.log(pr[np.arange(len(A)), A] + 1e-8)
            ratio = np.exp(lp - LOGP)
            clipped = np.clip(ratio, 1-eps, 1+eps)
            use_unclipped = (ratio*adv <= clipped*adv).astype(float)   # min 选了未裁剪项
            dlp = -(use_unclipped * ratio * adv) / len(A)              # 上升->对负目标下降
            onehot = np.zeros_like(pr); onehot[np.arange(len(A)), A] = 1.0
            dlogits = dlp[:,None] * (onehot - pr)
            ent = -(pr*np.log(pr+1e-8)).sum(1)
            dlogits += (-beta_ent) * pr*((-(np.log(pr+1e-8))) - ent[:,None]) / len(A)
            W1,b1,W2,b2 = pol; x,z1,hh = cache
            dW2 = dlogits.T@hh; db2 = dlogits.sum(0)
            dz1 = (dlogits@W2)*(1-hh**2); dW1 = dz1.T@x; db1 = dz1.sum(0)
            adam_step(pol, [dW1,db1,dW2,db2], pa, 3e-4)
            vp, vc = mlp_forward(val, S); vp = vp[:,0]
            dv = ((vp - ret)/len(ret))[:,None]
            W1,b1,W2,b2 = val; x,z1,hh = vc
            dW2v = dv.T@hh; db2v = dv.sum(0)
            dz1v = (dv@W2)*(1-hh**2); dW1v = dz1v.T@x; db1v = dz1v.sum(0)
            adam_step(val, [dW1v,db1v,dW2v,db2v], va, 1e-3)
        hist.append(np.mean(ep_rets))
    return pol, hist

pol, hist = train_ppo(seed=0, iters=30)
print('每5迭代平均回合长度:', [round(np.mean(hist[i:i+5]),1) for i in range(0,30,5)])
start = np.mean(hist[:3]); end = np.mean(hist[-5:])
print(f'起步存活 ~{start:.0f} 步 -> 训练后 ~{end:.0f} 步 (满分200)')
assert end > 120, f'PPO 应学会平衡(>120步)，实得 {end:.0f}'
assert end > start + 60, 'PPO 应显著提升存活步数'
print('✅ PPO 学会平衡杆！存活步数从 ~20 爬到 150+（固定 seed 可复现）')

## 5 · SAC 组件①：tanh 压缩高斯 + log-prob 修正

连续动作下，策略输出高斯 `N(μ, σ)`，再 `tanh` 压到 `[-1,1]`。**压缩改变了密度，必须修正**：
`log π(a) = log N(u; μ,σ) - Σ log(1 - tanh(u)²)`，`u` 是压缩前的样本。漏了修正项，熵估计就错。

In [ ]:
def squashed_gaussian(mu, log_sigma, eps):
    '''重参数化: u = μ + σ·eps; a = tanh(u)。返回 (动作 a, log_prob)。'''
    sigma = np.exp(log_sigma)
    u = mu + sigma * eps                                  # 重参数化(梯度可穿过)
    a = np.tanh(u)
    log_normal = -0.5*(((u-mu)/sigma)**2 + 2*log_sigma + np.log(2*np.pi))
    log_det_jac = np.log(1 - a**2 + 1e-6)                 # tanh 的雅可比修正
    log_prob = (log_normal - log_det_jac).sum(-1)
    return a, log_prob

mu = np.array([[0.0, 0.5]]); log_sigma = np.array([[0.0, -0.3]])
eps = np.array([[0.3, -0.2]])
a, logp = squashed_gaussian(mu, log_sigma, eps)
print('压缩后动作 a =', np.round(a, 3), ' (应在 (-1,1))')
print('log π(a) =', round(float(logp[0]), 3))
assert np.all(np.abs(a) < 1.0), 'tanh 把动作压进 (-1,1)'
# 验证修正项的作用：tanh 把宽 u 压成窄 a，a 空间密度更高 -> log-prob 因修正而增大
logp_nocorr = (-0.5*(((u_chk:=mu+np.exp(log_sigma)*eps)-mu)/np.exp(log_sigma))**2
               -0.5*(2*log_sigma + np.log(2*np.pi))).sum(-1)
assert logp[0] > logp_nocorr[0], 'tanh 雅可比修正(减去 log(1-a²)<0)使 log-prob 增大'
diff = float(logp[0] - logp_nocorr[0])
print(f'修正项贡献 = +{diff:.3f} (= -Σ log(1-a²) > 0)')
print('✅ 压缩高斯 + 雅可比修正正确（漏修正 -> 熵估计错 -> 温度调节失灵）')

## 6 · SAC 组件②③：双 Q 取 min + 温度自调 + 软更新

- **双 Q 取 min**：目标用 `min(Q1, Q2)` 压过估计；
- **温度自调**：`α` 通过约束维持目标熵 `H̄ = -dim(A)`；
- **软更新**：`θ̄ ← τθ + (1-τ)θ̄`。

我们实现 SAC 的目标计算与温度梯度，验证其逻辑（不做完整连续训练——部件正确是关键）。

In [ ]:
def sac_critic_target(r, done, q1_next, q2_next, logp_next, alpha, gamma=0.99):
    '''软 Bellman 目标：y = r + γ(1-done)(min(Q1,Q2) - α·logπ(a'|s'))。'''
    min_q = np.minimum(q1_next, q2_next)
    return r + gamma * (1 - done) * (min_q - alpha * logp_next)

def temperature_grad(log_alpha, logp, target_entropy):
    '''α 损失 = -α·(logπ + H̄) 的对 log_alpha 梯度（要最小化它）。'''
    alpha = np.exp(log_alpha)
    return -(alpha * (logp + target_entropy)).mean()

def soft_update(target, online, tau=0.005):
    return [tau*o + (1-tau)*t for t, o in zip(target, online)]

# 双 Q：目标取 min（压过估计）
r = np.array([1.0, 0.0]); done = np.array([0.0, 1.0])
q1n = np.array([2.0, 5.0]); q2n = np.array([1.5, 4.0]); logpn = np.array([-1.0, -1.0])
y = sac_critic_target(r, done, q1n, q2n, logpn, alpha=0.2)
assert np.isclose(y[1], 0.0), 'done=1 时目标=r(无未来)'
# 用了 min(Q1,Q2)=1.5 而非 max
assert np.isclose(y[0], 1.0 + 0.99*(min(2.0,1.5) - 0.2*(-1.0))), '应用 min(Q1,Q2)+熵项'
print('SAC critic 目标 y =', np.round(y, 3), '(用 min(Q1,Q2) 压过估计)')

# 温度调节方向：策略熵高于目标 -> α 应降；低于目标 -> α 应升
# 经验熵 ≈ -logp。目标熵 H̄=-1，即希望 logp≈1。
target_ent = -1.0           # dim(A)=1 -> 目标熵 = -1
logp_high_entropy = np.array([-3.0, -3.0])   # logπ 很负 -> 熵高(>目标)
g_high = temperature_grad(0.0, logp_high_entropy, target_ent)
logp_low_entropy = np.array([3.0, 3.0])      # logπ 很正 -> 熵低(<目标)
g_low = temperature_grad(0.0, logp_low_entropy, target_ent)
# grad = -α(logp+H̄)。梯度下降 log_alpha: grad>0 则 α↓，grad<0 则 α↑
# 熵高: logp+H̄=-4<0 -> grad>0 -> α↓(少奖励随机)；熵低: logp+H̄=2>0 -> grad<0 -> α↑(多鼓励探索)
assert g_high > 0 and g_low < 0, '熵高->α降(梯度正), 熵低->α升(梯度负)'
# 软更新
tgt = [np.ones(3)]; on = [np.array([2.,0.,4.])]
new = soft_update(tgt, on, tau=0.01)
assert np.allclose(new[0], [1.01, 0.99, 1.03])
print('✅ SAC 三大件逻辑正确：min双Q压高估、温度负反馈调节、软更新平滑目标')

---
## ✏️ 练习 1：PPO 比值与裁剪

实现 `importance_ratio_and_clip(logp_new, logp_old, eps)`，返回 `(ratio, clipped_ratio)`。
`ratio = exp(logp_new - logp_old)`，`clipped_ratio = clip(ratio, 1-eps, 1+eps)`。

In [ ]:
def importance_ratio_and_clip(logp_new, logp_old, eps=0.2):
    # TODO: ratio = exp(差)；clipped = np.clip(ratio, 1-eps, 1+eps)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
lo = np.array([-1.0, -1.0, -1.0])
ln = np.array([-1.0, -1.0 + np.log(2.0), -1.0 + np.log(0.5)])   # 比值 1, 2, 0.5
ratio, clipped = importance_ratio_and_clip(ln, lo, eps=0.2)
assert np.allclose(ratio, [1.0, 2.0, 0.5], atol=1e-6)
assert np.allclose(clipped, [1.0, 1.2, 0.8]), '应裁剪到 [0.8,1.2]'
print('ratio=', np.round(ratio,2), ' clipped=', np.round(clipped,2))
print('✅ 练习 1 通过：重要性比 + 裁剪')

## ✏️ 练习 2：GAE 的 λ 极限

实现 `gae_lambda_extremes(rewards, values, dones, gamma)`，返回 `(adv_lam0, adv_lam1)`：分别是 `λ=0`（单步 TD）与 `λ=1`（蒙特卡洛优势）的优势。复用 `compute_gae`。

In [ ]:
def gae_lambda_extremes(rewards, values, dones, gamma=0.99):
    # TODO: 用 compute_gae 分别传 lam=0.0 和 lam=1.0
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
rew = np.array([1.0, 1.0, 1.0]); val = np.array([0.5, 0.5, 0.5, 0.0]); done = np.array([0.,0.,1.])
a0, a1 = gae_lambda_extremes(rew, val, done, gamma=0.99)
# λ=0: 单步 TD δ_t
assert np.allclose(a0, rew + 0.99*val[1:]*(1-done) - val[:3], atol=1e-6), 'λ=0 应为单步TD'
# λ=1: 蒙特卡洛(折扣回报 - V)，方差最大、偏差最小
assert a1[0] > a0[0] or a1[0] < a0[0] or np.isclose(a1[0], a0[0])  # 只要能算出
assert a1.shape == (3,)
print('λ=0 优势=', np.round(a0,3), '\nλ=1 优势=', np.round(a1,3))
print('✅ 练习 2 通过：λ 在偏差(低)/方差(高)间插值的两个极限')

## ✏️ 练习 3：策略熵

实现 `policy_entropy(probs)`：给定动作概率 `(B, nA)`，返回每个状态的熵 `H = -Σ_a p log p`（长度 B）。
熵是 PPO/SAC 的探索正则项。

In [ ]:
def policy_entropy(probs):
    # TODO: H = -sum(p * log p) 沿动作维；用 +1e-8 防 log(0)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
uniform = np.array([[0.5, 0.5]])           # 最大熵(2动作)
deterministic = np.array([[0.999, 0.001]]) # 近零熵
H_u = policy_entropy(uniform); H_d = policy_entropy(deterministic)
assert np.isclose(H_u[0], np.log(2), atol=1e-3), '均匀分布熵=log(2)'
assert H_d[0] < 0.05, '近确定性分布熵≈0'
assert H_u[0] > H_d[0], '均匀比确定性熵大'
print(f'均匀熵={H_u[0]:.3f} (=log2={np.log(2):.3f}), 确定性熵={H_d[0]:.4f}')
print('✅ 练习 3 通过：熵在均匀时最大、确定时最小（鼓励探索）')

## ✏️ 练习 4：SAC 软更新（Polyak）

实现 `polyak_update(target_params, online_params, tau)`：逐参数 `θ̄ ← τθ + (1-τ)θ̄`，返回新的 target 参数列表。`tau` 小 → target 变化慢（更稳）。

In [ ]:
def polyak_update(target_params, online_params, tau=0.005):
    # TODO: 对每对 (t, o) 计算 tau*o + (1-tau)*t，返回列表
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
tgt = [np.zeros((2,2)), np.zeros(2)]
onl = [np.ones((2,2)), np.full(2, 4.0)]
new = polyak_update(tgt, onl, tau=0.25)
assert np.allclose(new[0], 0.25), 'θ̄ = 0.25*1 + 0.75*0'
assert np.allclose(new[1], 1.0), '0.25*4 + 0.75*0 = 1.0'
# tau=0 应不变, tau=1 应完全跟随
assert np.allclose(polyak_update(tgt, onl, tau=0.0)[0], 0.0)
assert np.allclose(polyak_update(tgt, onl, tau=1.0)[0], 1.0)
print('✅ 练习 4 通过：Polyak 软更新（τ 控制 target 跟随速度）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1
def importance_ratio_and_clip(logp_new, logp_old, eps=0.2):
    ratio = np.exp(logp_new - logp_old)
    return ratio, np.clip(ratio, 1 - eps, 1 + eps)

In [ ]:
# 练习 2
def gae_lambda_extremes(rewards, values, dones, gamma=0.99):
    a0, _ = compute_gae(rewards, values, dones, gamma=gamma, lam=0.0)
    a1, _ = compute_gae(rewards, values, dones, gamma=gamma, lam=1.0)
    return a0, a1

In [ ]:
# 练习 3
def policy_entropy(probs):
    return -(probs * np.log(probs + 1e-8)).sum(axis=-1)

In [ ]:
# 练习 4
def polyak_update(target_params, online_params, tau=0.005):
    return [tau*o + (1-tau)*t for t, o in zip(target_params, online_params)]

---
## 🧪 真实数据胶囊：PPO 与 SAC 的真实超参

下面是 **PPO（Schulman 2017 / CleanRL）与 SAC（Haarnoja 2018）** 的**真实**标准超参。用它们体会两类算法的不同节奏与那些「implementation matters」的细节。

In [ ]:
PPO_HP = dict(
    clip_eps=0.2, gamma=0.99, gae_lambda=0.95,
    lr=3e-4, epochs=10, minibatches=32, ent_coef=0.0,
    value_coef=0.5, max_grad_norm=0.5, normalize_advantage=True,
)
SAC_HP = dict(
    gamma=0.99, tau=0.005, lr=3e-4, batch=256,
    replay=1_000_000, target_entropy='-dim(A)', twin_q=True, auto_temperature=True,
)
print('PPO 真实超参:'); [print(f'  {k:20s}={v}') for k,v in PPO_HP.items()]
print('SAC 真实超参:'); [print(f'  {k:20s}={v}') for k,v in SAC_HP.items()]
# 关键对照：PPO clip_eps≈0.2、优势归一化几乎必开；SAC tau 很小(0.005)做软更新
assert PPO_HP['clip_eps'] == 0.2 and PPO_HP['normalize_advantage'] is True
assert SAC_HP['tau'] < 0.01 and SAC_HP['twin_q'] is True
print('\n✅ 我们的 toy PPO 用了相同的 clip_eps、GAE、优势归一化 —— 只是规模与网络更小')

**🧪 胶囊练习**：实现 `effective_kl_bound(clip_eps)`：PPO 的 clip 把比值限制在 `[1-ε, 1+ε]`，对应一个近似的「每步策略变化上限」。返回比值偏离 1 的最大幅度 `ε`（即 `clip_eps`），并验证它对应小信赖域。

In [ ]:
def effective_kl_bound(clip_eps):
    # TODO: clip 把比值限制在 1±clip_eps，返回这个偏离幅度 clip_eps
    raise NotImplementedError

In [ ]:
# 自测
assert abs(effective_kl_bound(0.2) - 0.2) < 1e-9
# clip 越小信赖域越紧(步子越保守)
assert effective_kl_bound(0.1) < effective_kl_bound(0.3)
print('clip_eps=0.2 -> 比值偏离上限 ±0.2 (近似小信赖域) ✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def effective_kl_bound(clip_eps):
    return clip_eps   # clip 把 π_new/π_old 限制在 1±clip_eps

### 小结
- **策略方法**直接优化 `π_θ`，天然支持连续/随机动作；策略梯度内核 = `∇log π · 优势`。
- **PPO**：用重要性比 `r` 复用旧数据，用 **clip(`r`, 1±ε)** 近似 TRPO 的信赖域——好动作封顶、坏动作托底，移除过大更新的激励。配优势归一化、熵正则、值损失。
- **最大熵 RL**：目标 = 回报 + α·熵，内生地鼓励探索与鲁棒；最优策略 ∝ exp(Q/α)。
- **SAC**：off-policy + 最大熵 + 三大件（**双 Q 取 min** 压高估、**tanh 压缩高斯 + 雅可比修正** 表示有界随机策略、**温度自调** 维持目标熵）+ 软更新。样本高效，连续控制主力。
- **PPO vs SAC**：并行模拟充足/含 RLHF → PPO；真实交互昂贵的连续控制 → SAC。

你已从零写出能学会平衡杆的 PPO、并实现 SAC 全部核心部件。下一站：**模块 03 · 离线强化学习** —— 连「交互」都不许，只给一个固定数据集。